# CrashLens: exploratory data analysis

Source: MoRTH *Road Accidents in India*, 2019 to 2024. One row is one State/UT, one category and one year, so every value is an aggregate count.

**Rules used throughout this notebook**

- Severity is always computed as `100 * sum(killed) / sum(accidents)`. A `severity_index` is never averaged.
- Each factor group is a separate breakdown of the same accidents. Totals are therefore taken from one group (`collision_type`), never summed across groups.
- Totals and severity ratios use all rows, including low-count rows, because their small counts still belong in the sums.
- Anything that uses `severity_level` (sections 5 and 6) uses the modelling dataset, with low-count rows dropped.
- Every chart is saved to `reports/figures` as PNG and PDF, and its numbers to `reports/tables` as CSV.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import chi2_contingency

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data import SEED, SEVERITY_ORDER, TOTALS_GROUP, load_data

np.random.seed(SEED)
FIG_DIR = ROOT / "reports" / "figures"
TAB_DIR = ROOT / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

# One palette for every figure: a single blue for magnitude, grey for context.
BLUE = "#2a78d6"
BLUE_LIGHT = "#b7d3f6"
ORDINAL = {"Low": "#86b6ef", "Medium": "#2a78d6", "High": "#104281"}
SEQ = LinearSegmentedColormap.from_list("blue_seq", ["#cde2fb", "#6da7ec", "#2a78d6", "#0d366b"])
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "font.family": "sans-serif", "font.size": 9,
    "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 9, "axes.labelcolor": INK2, "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.grid": False, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "legend.frameon": False, "legend.fontsize": 8,
    "lines.linewidth": 2, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def save_fig(fig, name):
    """Save a figure to reports/figures as PNG (300 dpi) and PDF."""
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    print(f"saved reports/figures/{name}.png and .pdf")


def severity(d):
    """Killed per 100 accidents, from summed counts."""
    return 100 * d["killed"].sum() / d["accidents"].sum()


def severity_table(d, by):
    """Summed accidents, summed killed and the resulting severity for each group."""
    t = d.groupby(by, observed=True)[["accidents", "killed"]].sum()
    t["severity"] = 100 * t["killed"] / t["accidents"]
    return t


def pretty(name):
    return name.replace("_", " ").capitalize()


def wrap(label, width=24):
    """Break a long category label at its slash so it does not crowd the plot."""
    return label.replace("/", "/\n", 1) if len(label) > width and "/" in label else label


raw = load_data(drop_low_count=False)   # all rows: used for summed totals and ratios
df = load_data()                        # modelling rows: used wherever severity_level is needed
tot = raw[raw["factor_group"] == TOTALS_GROUP]
LATEST = int(raw["year"].max())
NATIONAL_ALL = severity(tot)
print(f"all rows: {len(raw):,} | modelling rows: {len(df):,} | national severity, all years: {NATIONAL_ALL:.2f}")

all rows: 11,752 | modelling rows: 9,332 | national severity, all years: 35.95


## 0. Why totals come from one factor group

The table shows accidents summed within each factor group, as a percentage of the `collision_type` total for that year.

In [2]:
coverage = raw.pivot_table(index="factor_group", columns="year", values="accidents", aggfunc="sum")
(coverage / coverage.loc[TOTALS_GROUP] * 100).round(1)

year,2019,2020,2021,2022,2023,2024
factor_group,,,,,,
area_type,100.0,100.0,100.0,100.0,100.0,100.0
collision_type,100.0,100.0,100.0,100.0,100.0,100.0
impacting_vehicle,100.0,100.0,100.0,100.0,100.0,100.0
junction_type,100.0,100.0,100.0,22.3,22.6,20.1
road_environment,100.0,100.0,100.0,100.0,100.0,41.2
road_feature,100.0,100.0,100.0,100.0,100.0,100.0
traffic_control,100.0,100.0,100.0,22.3,22.6,20.1
traffic_violation,100.0,100.0,100.0,100.0,100.0,100.0
weather,100.0,100.0,100.0,100.0,100.0,100.0


**Finding:** Six of the nine factor groups each reproduce the full national total in every year, so totals must come from one group; `junction_type` and `traffic_control` cover only about 20 to 23% of accidents from 2022 (their "Others" category is no longer reported) and `road_environment` covers 41% in 2024, so comparisons across years for those three groups need care.

## 1. National accidents, deaths and severity per year

In [3]:
national = severity_table(tot, "year")
national.round(2).to_csv(TAB_DIR / "national_by_year.csv")

panels = [("accidents", "Accidents (thousands)", 1e3, "{:,.0f}k"),
          ("killed", "Persons killed (thousands)", 1e3, "{:,.0f}k"),
          ("severity", "Severity (killed per 100 accidents)", 1, "{:.1f}")]
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))
for ax, (col, title, div, fmt) in zip(axes, panels):
    y = national[col] / div
    pad = (y.max() - y.min()) * 0.3
    ax.set_ylim(y.min() - pad, y.max() + pad)
    ax.axvspan(2019.6, 2020.4, color="#f0efec", zorder=0)
    ax.text(2020, y.max() + pad * 0.9, "COVID-19\nyear", ha="center", va="top", fontsize=7.5, color=INK2)
    ax.plot(y.index, y.values, color=BLUE, marker="o", markersize=6.5, markeredgecolor="white",
            markeredgewidth=1.5, zorder=3)
    for yr in (y.index.min(), 2020, y.index.max()):
        if col == "severity":   # the line rises through 2019 and 2020, so keep labels off it
            offset = {y.index.min(): (0, -15), 2020: (-14, 5)}.get(yr, (0, 8))
        else:
            offset = (0, -15) if yr == 2020 else (0, 8)
        ax.annotate(fmt.format(y[yr]), (yr, y[yr]), textcoords="offset points",
                    xytext=offset, ha="center", fontsize=8, color=INK)
    ax.set_title(title)
    ax.set_xticks(y.index)
    ax.set_xlim(2018.6, 2024.4)
    ax.grid(axis="y")
    ax.tick_params(length=0)
fig.suptitle("India, national road accident totals, 2019 to 2024", x=0.01, ha="left", fontsize=11, fontweight="bold")
fig.tight_layout()
save_fig(fig, "fig01_national_trend")
plt.show()
national.round(2)

saved reports/figures/fig01_national_trend.png and .pdf


C:\Users\Tushar\AppData\Local\Temp\ipykernel_46160\2125865782.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,accidents,killed,severity
year,,,
2019,449002,151113,33.66
2020,366138,131714,35.97
2021,412432,153972,37.33
2022,461312,168491,36.52
2023,480583,172890,35.98
2024,487707,177175,36.33


**Finding:** Accidents fell 18% and deaths 13% in the COVID year 2020, then both rose past their 2019 levels by 2022; because deaths fell less than accidents, severity jumped from 33.7 to 36.0 in 2020, peaked at 37.3 in 2021 and has stayed near 36 since, so crashes are deadlier than before the pandemic.

## 2. Severity by factor value within each factor group

National level, all years combined. The vertical line is the national severity for all years combined.

In [4]:
by_value = severity_table(raw, ["factor_group", "factor_value"])
by_value.round(2).to_csv(TAB_DIR / "severity_by_factor_value.csv")

groups = ["collision_type", "impacting_vehicle", "road_feature", "junction_type", "traffic_control",
          "traffic_violation", "weather", "road_environment", "area_type"]
slots = [9, 6, 5]   # bar slots per row of panels, so bars keep one thickness across the figure
fig, axes = plt.subplots(3, 3, figsize=(13, 8.6), gridspec_kw={"height_ratios": [s + 1 for s in slots]})
xmax = by_value["severity"].max() * 1.16
for k, (ax, g) in enumerate(zip(axes.ravel(), groups)):
    t = by_value.loc[g].sort_values("severity")
    n_slots = slots[k // 3]
    ax.barh([wrap(v) for v in t.index], t["severity"], height=0.62, color=BLUE, zorder=2)
    ax.axvline(NATIONAL_ALL, color=INK2, linewidth=1, zorder=1.5)
    for i, v in enumerate(t["severity"]):
        ax.text(v + xmax * 0.012, i, f"{v:.1f}", va="center", fontsize=7.5, color=INK, zorder=4,
                bbox=dict(facecolor="white", edgecolor="none", pad=0.6))
    ax.set_title(pretty(g))
    ax.set_xlim(0, xmax)
    ax.set_ylim(len(t) - n_slots - 0.6, len(t) + 0.2)   # bars sit at the top of the panel
    ax.grid(axis="x")
    ax.tick_params(length=0)
    ax.spines["bottom"].set_visible(False)
axes[0, 0].text(NATIONAL_ALL, 8.42, f" national {NATIONAL_ALL:.1f}", fontsize=7.5, color=INK2, va="bottom", ha="left")
for ax in axes[-1]:
    ax.set_xlabel("Killed per 100 accidents")
fig.suptitle("Severity by factor value, India, 2019 to 2024 combined", x=0.01, ha="left", fontsize=11, fontweight="bold")
fig.tight_layout()
save_fig(fig, "fig02_severity_by_factor_value")
plt.show()

saved reports/figures/fig02_severity_by_factor_value.png and .pdf


C:\Users\Tushar\AppData\Local\Temp\ipykernel_46160\1829930229.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Single-vehicle and escape crashes are the deadliest (run off road 46.1, vehicle overturn 45.8, hit and run 45.3 against 28.7 for hit from side), rural roads are far deadlier than urban ones (40.1 against 29.4), signal-controlled sites are the safest (24.3), and over-speeding sits slightly below the national 36.0 because it accounts for 71% of all accidents and so defines the average.

## 3. Weather, road feature and collision type severity by year

In [5]:
trend_groups = ["weather", "road_feature", "collision_type"]
by_year = severity_table(raw[raw["factor_group"].isin(trend_groups)], ["factor_group", "factor_value", "year"])
by_year.round(2).to_csv(TAB_DIR / "severity_by_factor_value_year.csv")
grids = {}
for g in trend_groups:
    m = by_year.loc[g]["severity"].unstack("year")
    grids[g] = m.loc[m.mean(axis=1).sort_values(ascending=False).index]
vmin = min(m.min().min() for m in grids.values())
vmax = max(m.max().max() for m in grids.values())

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), gridspec_kw={"wspace": 0.55})
for ax, g in zip(axes, trend_groups):
    m = grids[g]
    im = ax.imshow(m.values, cmap=SEQ, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(m.shape[1]), m.columns)
    ax.set_yticks(range(m.shape[0]), [wrap(v) for v in m.index])
    ax.set_xticks(np.arange(-0.5, m.shape[1]), minor=True)
    ax.set_yticks(np.arange(-0.5, m.shape[0]), minor=True)
    ax.grid(which="minor", color="white", linewidth=2)
    ax.tick_params(which="both", length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            v = m.values[i, j]
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                    color="white" if (v - vmin) / (vmax - vmin) > 0.45 else INK)
    ax.set_title(pretty(g))
cb = fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02)
cb.set_label("Killed per 100 accidents")
cb.outline.set_visible(False)
fig.suptitle("Severity by year, India (rows sorted by mean severity)", x=0.01, y=1.02, ha="left", fontsize=11, fontweight="bold")
save_fig(fig, "fig03_severity_by_year_heatmap")
plt.show()
change = pd.concat({g: pd.DataFrame({"2019": m[2019], str(LATEST): m[LATEST], "change": m[LATEST] - m[2019]})
                    for g, m in grids.items()})
change.round(1)

saved reports/figures/fig03_severity_by_year_heatmap.png and .pdf


C:\Users\Tushar\AppData\Local\Temp\ipykernel_46160\1162001697.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


2019  2024  change
               factor_value                                             
weather        Hail/Sleet                             50.4  44.5    -5.8
               Foggy & Misty                          39.9  44.5     4.6
               Others                                 42.8  45.4     2.6
               Rainy                                  35.8  35.9     0.1
               Sunny/Clear                            31.4  34.5     3.1
road_feature   Ongoing road works/Under construction  38.5  50.0    11.6
               Culvert                                38.8  39.7     0.8
               Steep grade                            38.4  44.3     5.9
               Potholes                               44.8  43.9    -0.9
               Bridge                                 37.9  40.7     2.8
               Curved road                            33.1  36.5     3.4
               Straight road                          32.9  36.0     3.2
               Others                                 34.1  32.0    -2.1
collision_type Vehicle overturn                       38.7  48.7    10.0
               Run off road                           44.2  46.7     2.5
               Hit and run                            42.2  49.6     7.5
               Fixed object                           39.6  46.3     6.7
               With parked vehicle                    38.2  40.1     1.9
               Head on collision                      30.6  33.6     3.0
               Others                                 30.6  32.9     2.3
               Hit from back                          30.9  32.8     2.0
               Hit from side                          28.2  28.2    -0.0

**Finding:** Severity rose between 2019 and 2024 for most categories, most sharply for road works (38.5 to 50.0), vehicle overturn (38.6 to 48.7) and hit and run (42.2 to 49.6); fog stayed consistently deadlier than clear weather (44.5 against 34.5 in 2024), while hit from side stayed flat at about 28 and the small hail/sleet category fell from 50.4 to 44.5.

## 4. Severity by region and state ranking for the latest year

Regions use all years combined. The state ranking uses the latest year only.

In [6]:
region = severity_table(tot, "region").sort_values("severity")
region.round(2).to_csv(TAB_DIR / "severity_by_region.csv")

fig, ax = plt.subplots(figsize=(6, 3.1))
ax.barh(region.index, region["severity"], height=0.6, color=BLUE, zorder=2)
ax.axvline(NATIONAL_ALL, color=INK2, linewidth=1, zorder=3)
ax.text(NATIONAL_ALL, len(region) - 0.45, f" national {NATIONAL_ALL:.1f}", fontsize=7.5, color=INK2, va="bottom")
for i, v in enumerate(region["severity"]):
    ax.text(v + 0.5, i, f"{v:.1f}", va="center", fontsize=8, color=INK)
ax.set_xlim(0, region["severity"].max() * 1.12)
ax.set_ylim(-0.7, len(region) - 0.05)
ax.set_xlabel("Killed per 100 accidents")
ax.set_title("Severity by region, 2019 to 2024 combined")
ax.grid(axis="x"); ax.tick_params(length=0); ax.spines["bottom"].set_visible(False)
fig.tight_layout()
save_fig(fig, "fig04a_severity_by_region")
plt.show()
region.round(2)

saved reports/figures/fig04a_severity_by_region.png and .pdf


C:\Users\Tushar\AppData\Local\Temp\ipykernel_46160\1680246952.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,accidents,killed,severity
region,,,
South,1100726,272607,24.77
Central,390774,110388,28.25
North-East,57840,23599,40.80
West,303545,130460,42.98
North,572277,279644,48.87
East,232012,138657,59.76


**Finding:** Severity differs by a factor of 2.4 across regions, from 24.8 in the South to 59.8 in the East, so geography separates severity far more strongly than any single road or crash factor.

In [7]:
latest = tot[tot["year"] == LATEST]
NATIONAL_LATEST = severity(latest)
states = severity_table(latest, "state")
states = states[states["accidents"] > 0].sort_values("severity")
states.round(2).sort_values("severity", ascending=False).to_csv(TAB_DIR / f"state_ranking_{LATEST}.csv")
SMALL = 500   # below this many accidents a state's ratio is unstable

fig, ax = plt.subplots(figsize=(7, 8.5))
colors = [BLUE if a >= SMALL else BLUE_LIGHT for a in states["accidents"]]
ax.barh(states.index, states["severity"], height=0.66, color=colors, zorder=2)
ax.axvline(NATIONAL_LATEST, color=INK2, linewidth=1, zorder=3)
ax.text(NATIONAL_LATEST, len(states) - 0.3, f" national {NATIONAL_LATEST:.1f}", fontsize=7.5, color=INK2, va="bottom")
for i, v in enumerate(states["severity"]):
    ax.text(v + 0.8, i, f"{v:.1f}", va="center", fontsize=7.5, color=INK)
ax.set_xlim(0, states["severity"].max() * 1.1)
ax.set_ylim(-0.8, len(states) + 0.4)
ax.set_xlabel("Killed per 100 accidents")
ax.set_title(f"State and UT ranking by severity, {LATEST}")
ax.grid(axis="x"); ax.tick_params(length=0); ax.spines["bottom"].set_visible(False)
handles = [plt.Rectangle((0, 0), 1, 1, color=BLUE), plt.Rectangle((0, 0), 1, 1, color=BLUE_LIGHT)]
ax.legend(handles, [f"{SMALL:,} or more accidents", f"Fewer than {SMALL:,} accidents (unstable ratio)"], loc="lower right")
fig.tight_layout()
save_fig(fig, f"fig04b_state_ranking_{LATEST}")
plt.show()
pd.concat({"highest": states.tail(5).iloc[::-1], "lowest": states.head(5)}).round(1)

saved reports/figures/fig04b_state_ranking_2024.png and .pdf


C:\Users\Tushar\AppData\Local\Temp\ipykernel_46160\4077898286.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


accidents  killed  severity
        state                                                 
highest Mizoram                          118     110      93.2
        Bihar                          11610    9347      80.5
        Jharkhand                       5196    4114      79.2
        Punjab                          6063    4759      78.5
        Meghalaya                        269     197      73.2
lowest  Kerala                         48834    3880       7.9
        Goa                             2682     286      10.7
        Jammu & Kashmir                 5808     831      14.3
        Puducherry                      1431     218      15.2
        Andaman & Nicobar Islands        135      29      21.5

**Finding:** In 2024 Bihar (80.5), Jharkhand (79.2) and Punjab (78.5) lead among states with at least 500 accidents while Kerala (7.9) and Goa (10.7) are lowest, a tenfold spread; Mizoram's 93.2 rests on only 118 accidents and should not be read as a stable rate.

## 5. Class balance of `severity_level` by year

Modelling rows only (low-count rows dropped).

In [8]:
counts = pd.crosstab(df["year"], df["severity_level"])[SEVERITY_ORDER]
share = counts.div(counts.sum(axis=1), axis=0) * 100
counts.assign(total=counts.sum(axis=1)).to_csv(TAB_DIR / "class_balance_by_year.csv")

fig, ax = plt.subplots(figsize=(7, 3.7))
bottom = np.zeros(len(share))
for lvl in SEVERITY_ORDER:
    ax.bar(share.index, share[lvl], bottom=bottom, width=0.62, color=ORDINAL[lvl], edgecolor="white",
           linewidth=2, label=lvl)
    for x, b, v in zip(share.index, bottom, share[lvl]):
        ax.text(x, b + v / 2, f"{v:.0f}%", ha="center", va="center", fontsize=8,
                color=INK if lvl == "Low" else "white")
    bottom += share[lvl].values
ax.set_ylim(0, 100)
ax.set_xticks(share.index)
ax.set_ylabel("Share of rows (%)")
ax.set_title("Class balance of severity level by year", pad=24)
ax.legend(ncol=3, loc="lower left", bbox_to_anchor=(-0.01, 0.99), handlelength=1, columnspacing=1.2)
ax.tick_params(length=0)
fig.tight_layout()
save_fig(fig, "fig05_class_balance_by_year")
plt.show()
pd.concat([counts, share.round(1).add_suffix(" %")], axis=1)

saved reports/figures/fig05_class_balance_by_year.png and .pdf


C:\Users\Tushar\AppData\Local\Temp\ipykernel_46160\4045504740.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


severity_level,Low,Medium,High,Low %,Medium %,High %
year,,,,,,
2019,505,479,637,31.2,29.5,39.3
2020,479,463,615,30.8,29.7,39.5
2021,501,524,571,31.4,32.8,35.8
2022,483,486,584,31.1,31.3,37.6
2023,468,475,602,30.3,30.7,39.0
2024,415,438,607,28.4,30.0,41.6


**Finding:** The classes are only mildly imbalanced and High is the largest class in every year (36 to 42% of rows, against 28 to 31% for Low and 30 to 33% for Medium), with High reaching its largest share in 2024, which falls in the test period.

## 6. Chi-square test of `severity_level` against `factor_value`, per factor group

Modelling rows only. Cramér's V is reported with each test because, with hundreds of rows per group, a p-value alone says little about the strength of an association.

In [9]:
rows = []
for g, d in df.groupby("factor_group"):
    table = pd.crosstab(d["factor_value"], d["severity_level"])
    chi2, p, dof, expected = chi2_contingency(table)
    n = table.values.sum()
    rows.append({"factor_group": g, "n_rows": n, "n_values": table.shape[0], "chi2": chi2, "dof": dof, "p_value": p,
                 "cramers_v": np.sqrt(chi2 / (n * (min(table.shape) - 1))), "min_expected": expected.min()})
chi = pd.DataFrame(rows).set_index("factor_group").sort_values("cramers_v", ascending=False)
chi.to_csv(TAB_DIR / "chi_square_severity_vs_factor_value.csv")

t = chi.sort_values("cramers_v")
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.barh([pretty(i) for i in t.index], t["cramers_v"], height=0.6, color=BLUE, zorder=2)
for i, (v, p) in enumerate(zip(t["cramers_v"], t["p_value"])):
    ax.text(v + 0.005, i, f"{v:.2f}" + ("" if p < 0.05 else "  (not significant)"), va="center", fontsize=8, color=INK)
ax.set_xlim(0, t["cramers_v"].max() * 1.18)
ax.set_xlabel("Cramér's V (0 = no association, 1 = perfect)")
ax.set_title("Association between severity level and factor value")
ax.grid(axis="x"); ax.tick_params(length=0); ax.spines["bottom"].set_visible(False)
fig.tight_layout()
save_fig(fig, "fig06_chi_square_cramers_v")
plt.show()
chi.style.format({"chi2": "{:.1f}", "p_value": "{:.2e}", "cramers_v": "{:.3f}", "min_expected": "{:.1f}"})

saved reports/figures/fig06_chi_square_cramers_v.png and .pdf


C:\Users\Tushar\AppData\Local\Temp\ipykernel_46160\3102211089.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,n_rows,n_values,chi2,dof,p_value,cramers_v,min_expected
factor_group,,,,,,,
area_type,412,2,34.7,2,2.93e-08,0.290,56.0
impacting_vehicle,1530,9,109.7,16,5.15e-16,0.189,32.9
road_environment,905,5,51.4,8,2.22e-08,0.168,41.3
collision_type,1665,9,94.2,16,4.15e-13,0.168,45.0
traffic_control,846,6,46.3,10,1.28e-06,0.165,29.5
traffic_violation,976,6,45.9,10,1.50e-06,0.153,31.6
weather,843,5,19.8,8,1.10e-02,0.108,27.9
junction_type,898,6,18.7,10,4.45e-02,0.102,32.7
road_feature,1257,8,13.6,14,4.77e-01,0.074,29.0


**Finding:** Severity level depends on factor value in eight of the nine groups at the 5% level, but every association is weak to moderate (Cramér's V from 0.10 to 0.29, strongest for area type), and road feature shows no significant association (p = 0.48); the p-values are optimistic because the same states recur across rows, so the rows are not independent.